In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

def rmse(a, p):
    return np.sqrt(np.mean((a - p) ** 2))

panel = pd.read_parquet("Panel.parquet")

In [2]:
HAR  = ["RV_daily", "RV_weekly", "RV_monthly"]
HARX = HAR + ["CDS_level", "CDS_change_5d", "VIX", "Yield_slope", "BAA10Y"]

# out of sample evaluation window
TEST_YEARS = [2020, 2021, 2022, 2023, 2024] 
HORIZONS   = ["Target_h1", "Target_h5", "Target_h22"]

# Hyperparameters tuned by time series CV within the training window.
RF_PARAMS  = dict(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1)
XGB_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05,
                  subsample=0.8, random_state=42, verbosity=0)

In [3]:
def walk_forward(panel, tgt):
    d = panel.dropna(subset=HARX + [tgt]).copy()
    preds = {m: [] for m in ["actual", "HAR", "HARX", "RF", "XGB"]}
    per_year = {}
    for ty in TEST_YEARS:
        tr = d[d["Date"].dt.year <  ty]
        te = d[d["Date"].dt.year == ty]
        ytr, a = tr[tgt].values, te[tgt].values
        preds["actual"].append(a)
        ph = np.clip(LinearRegression().fit(tr[HAR].values,  ytr).predict(te[HAR].values),  0, None)
        px = np.clip(LinearRegression().fit(tr[HARX].values, ytr).predict(te[HARX].values), 0, None)
        pr = np.clip(RandomForestRegressor(**RF_PARAMS).fit(tr[HARX].values, ytr).predict(te[HARX].values), 0, None)
        pg = np.clip(xgb.XGBRegressor(**XGB_PARAMS).fit(tr[HARX].values, ytr).predict(te[HARX].values), 0, None)
        for m, p in [("HAR", ph), ("HARX", px), ("RF", pr), ("XGB", pg)]:
            preds[m].append(p)
        per_year[ty] = {m: rmse(a, p) for m, p in
                        [("HAR", ph), ("HARX", px), ("RF", pr), ("XGB", pg)]}
    actual = np.concatenate(preds["actual"])
    pooled = {m: rmse(actual, np.concatenate(preds[m]))
              for m in ["HAR", "HARX", "RF", "XGB"]}
    return pooled, per_year

In [4]:
pooled_results, peryear_results = {}, {}
for tgt in HORIZONS:
    pooled, per_year = walk_forward(panel, tgt)
    pooled_results[tgt]  = pooled
    peryear_results[tgt] = per_year
    winner = min(pooled, key=pooled.get)
    print(f"\n{tgt}  winner: {winner}")
    print("Pooled RMSE:", {m: round(v, 5) for m, v in pooled.items()})
    print(f"{'Year':<6}{'HAR':>9}{'HARX':>9}{'RF':>9}{'XGB':>9}  Winner")
    for y in TEST_YEARS:
        v = per_year[y]
        w = min(v, key=v.get)
        print(f"{y:<6}" + "".join(f"{v[m]:>9.4f}" for m in ['HAR','HARX','RF','XGB']) + f"  {w}")


Target_h1  winner: HARX
Pooled RMSE: {'HAR': np.float64(0.91489), 'HARX': np.float64(0.91223), 'RF': np.float64(1.02763), 'XGB': np.float64(1.11167)}
Year        HAR     HARX       RF      XGB  Winner
2020     1.9847   1.9760   2.0098   2.1338  HARX
2021     0.2103   0.2137   0.2168   0.2222  HAR
2022     0.3417   0.3538   1.0438   1.2304  HAR
2023     0.2015   0.2027   0.2275   0.2010  XGB
2024     0.3457   0.3447   0.3700   0.3461  HARX

Target_h5  winner: HARX
Pooled RMSE: {'HAR': np.float64(0.49997), 'HARX': np.float64(0.49544), 'RF': np.float64(0.66611), 'XGB': np.float64(0.65434)}
Year        HAR     HARX       RF      XGB  Winner
2020     1.0902   1.0750   1.0448   1.0784  RF
2021     0.1101   0.1209   0.1148   0.1097  XGB
2022     0.1720   0.1912   1.0429   0.9734  HAR
2023     0.0998   0.1044   0.1180   0.1019  HAR
2024     0.1695   0.1677   0.1856   0.1760  HARX

Target_h22  winner: HARX
Pooled RMSE: {'HAR': np.float64(0.35052), 'HARX': np.float64(0.34337), 'RF': np.float64(

In [5]:
# Pooled RMSE
pooled_df = pd.DataFrame(pooled_results).T
pooled_df.index.name = "Horizon"
pooled_df.to_csv("Pooled_RMSE.csv")

# Per year RMSE
rows = []
for tgt, yr in peryear_results.items():
    for y, v in yr.items():
        rows.append({"Horizon": tgt, "Year": y, **v})
peryear_df = pd.DataFrame(rows)
peryear_df.to_csv("Peryear_RMSE.csv", index=False)

print("\nPooled RMSE:")
print(pooled_df.round(5))


Pooled RMSE:
                HAR     HARX       RF      XGB
Horizon                                       
Target_h1   0.91489  0.91223  1.02763  1.11167
Target_h5   0.49997  0.49544  0.66611  0.65434
Target_h22  0.35052  0.34337  0.59005  0.71740


In [6]:
def mae(a, p):
    return np.mean(np.abs(a - p))

pooled_mae_all, peryear_mae_all = {}, {}
for tgt in HORIZONS:
    d = panel.dropna(subset=HARX + [tgt]).copy()
    preds = {m: [] for m in ["actual", "HAR", "HARX", "RF", "XGB"]}
    per_year_mae = {}
    for ty in TEST_YEARS:
        tr = d[d["Date"].dt.year <  ty]
        te = d[d["Date"].dt.year == ty]
        ytr, a = tr[tgt].values, te[tgt].values
        preds["actual"].append(a)
        ph = np.clip(LinearRegression().fit(tr[HAR].values,  ytr).predict(te[HAR].values),  0, None)
        px = np.clip(LinearRegression().fit(tr[HARX].values, ytr).predict(te[HARX].values), 0, None)
        pr = np.clip(RandomForestRegressor(**RF_PARAMS).fit(tr[HARX].values, ytr).predict(te[HARX].values), 0, None)
        pg = np.clip(xgb.XGBRegressor(**XGB_PARAMS).fit(tr[HARX].values, ytr).predict(te[HARX].values), 0, None)
        for m, p in [("HAR", ph), ("HARX", px), ("RF", pr), ("XGB", pg)]:
            preds[m].append(p)
        per_year_mae[ty] = {m: mae(a, p) for m, p in [("HAR", ph), ("HARX", px), ("RF", pr), ("XGB", pg)]}
    actual = np.concatenate(preds["actual"])
    pooled_mae_all[tgt] = {m: mae(actual, np.concatenate(preds[m])) for m in ["HAR", "HARX", "RF", "XGB"]}
    peryear_mae_all[tgt] = per_year_mae

In [7]:
mae_pooled_df = pd.DataFrame(pooled_mae_all).T
mae_pooled_df.index.name = "Horizon"
mae_pooled_df.to_csv("MAE_Pooled.csv")

peryear_mae_rows = []
for tgt, yr in peryear_mae_all.items():
    for y, v in yr.items():
        peryear_mae_rows.append({"Horizon": tgt, "Year": y, **v})
pd.DataFrame(peryear_mae_rows).to_csv("MAE_Peryear.csv", index=False)

print("Pooled MAE:")
print(mae_pooled_df.round(5))

Pooled MAE:
                HAR     HARX       RF      XGB
Horizon                                       
Target_h1   0.15237  0.16680  0.24769  0.21268
Target_h5   0.10979  0.12436  0.17876  0.16857
Target_h22  0.09384  0.10861  0.24410  0.28621
